# PortfolioRiskDiagnosis Review

This notebook is a review surface for the current diagnosis engine.

It is organized around five user-facing questions:

1. Overall Diagnosis Summary
2. Top Risk Categories (Ranked)
3. Top Risk Drivers
4. Supporting Evidence
5. Confidence and Completeness Flags

The goal is to make the diagnosis understandable enough for product review, UI brainstorming, and trust checks.


## Philosophy

This notebook is intentionally object-first.

We still use dataframes for display, but the logic flows through the typed `PortfolioRiskDiagnosis` object from [diagnosis.py](/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/portfolio_analyzer/diagnosis.py).

That keeps the review aligned with the architecture we actually want to build forward from.


In [1]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DIAGNOSIS_DIR = ROOT / 'data' / 'processed' / 'diagnosis'
OUTPUT_PATH = DIAGNOSIS_DIR / 'portfolio_risk_diagnosis_enriched.json'

import portfolio_analyzer.diagnosis as diagnosis_module
importlib.reload(diagnosis_module)

portfolio_risk_diagnosis_from_saved_artifacts = diagnosis_module.portfolio_risk_diagnosis_from_saved_artifacts
DiagnosisConcern = diagnosis_module.DiagnosisConcern

diagnosis = portfolio_risk_diagnosis_from_saved_artifacts(DIAGNOSIS_DIR)

print('Diagnosis directory:', DIAGNOSIS_DIR)
print('Run ID:', diagnosis.run_id)
print('Observed risk:', diagnosis.observed_risk_score, diagnosis.observed_risk_band)
print('Stated risk:', diagnosis.stated_risk_score, diagnosis.stated_risk_band)
print('Confidence:', diagnosis.confidence_band)


Diagnosis directory: /Users/sagnikrana/Documents/GitHub/portfolio-analyzer/data/processed/diagnosis
Run ID: 20260419_162540
Observed risk: 53.4 Moderate
Stated risk: 60.0 Moderate
Confidence: High


In [2]:

def _fmt_pct(value):
    if value is None or pd.isna(value):
        return "-"
    return f"{float(value):.1%}"


def _fmt_score(value):
    if value is None or pd.isna(value):
        return "-"
    return f"{float(value):.1f}"


def _join_or_dash(values):
    values = [str(value) for value in (values or []) if value not in {None, ''}]
    return ', '.join(values) if values else '-'


def confidence_interpretation(diagnosis):
    coverage = diagnosis.data_coverage.model_dump()
    available_count = sum(1 for value in coverage.values() if value)
    if diagnosis.confidence_band == 'High' and available_count >= 7:
        return 'High confidence: multiple independent signals agree and the diagnosis has broad source coverage.'
    if diagnosis.confidence_band in {'High', 'Medium'}:
        return 'Medium confidence: several signals agree, but some areas still need deeper modeling.'
    return 'Low confidence: evidence is still thin, so the diagnosis should be treated as preliminary.'


def build_overall_summary_df(diagnosis):
    return pd.DataFrame([
        {
            'observed_risk_score': diagnosis.observed_risk_score,
            'observed_risk_band': diagnosis.observed_risk_band,
            'stated_risk_score': diagnosis.stated_risk_score,
            'stated_risk_band': diagnosis.stated_risk_band,
            'alignment_status': diagnosis.alignment,
            'overall_confidence': diagnosis.confidence_band,
            'confidence_interpretation': confidence_interpretation(diagnosis),
            'analysis_start': diagnosis.analysis_start,
            'analysis_end': diagnosis.analysis_end,
            'benchmark_symbol': diagnosis.benchmark_symbol,
        }
    ])


def build_fake_dataset_overview_markdown(diagnosis):
    top_concerns = ', '.join(concern.label for concern in diagnosis.top_concerns[:3]) or 'No major concerns identified yet'
    top_holdings = ', '.join(
        f"{driver.ticker} ({driver.primary_reason_label.lower()})"
        for driver in diagnosis.top_holding_drivers[:3]
        if driver.primary_reason_label
    ) or 'No holding drivers yet'
    top_sectors = ', '.join(driver.sector for driver in diagnosis.top_sector_drivers[:2]) or 'No sector drivers yet'
    lines = [
        '### Current fake-dataset diagnosis',
        '',
        (
            f"For the current saved fake-dataset run, the portfolio looks **{diagnosis.alignment.lower()}** with the user's "
            f"stated risk tolerance: observed risk is **{diagnosis.observed_risk_score:.1f}/100 ({diagnosis.observed_risk_band})** "
            f"versus stated risk **{diagnosis.stated_risk_score:.1f}/100 ({diagnosis.stated_risk_band})**."
        ),
        '',
        (
            f"The strongest current concerns are **{top_concerns}**. The main holding-level drivers right now are "
            f"**{top_holdings}**, while the sector view is led by **{top_sectors}**."
        ),
        '',
        (
            'This means the portfolio is not flashing a pure "overall risk mismatch" warning right now. '
            'Instead, the diagnosis is saying: the portfolio still has **specific fragility pockets** that deserve '
            'attention, especially in the names and sectors listed below.'
        ),
    ]
    return "\n".join(lines)


def build_top_risk_categories_df(diagnosis):
    return pd.DataFrame([
        {
            'label': concern.label,
            'base_severity_score': concern.base_severity_score,
            'external_adjustment_score': concern.external_adjustment_score,
            'severity_score': concern.severity_score,
            'severity_band': concern.severity_band,
            'summary': concern.summary,
            'external_reinforcement': _join_or_dash(concern.adjustment_reasons),
            'related_tickers': _join_or_dash(concern.related_tickers),
            'related_sectors': _join_or_dash(concern.related_sectors),
        }
        for concern in diagnosis.top_concerns
    ])


def build_top_concern_markdown(diagnosis):
    lines = ["### Why these categories are ranked the way they are", ""]
    for concern in diagnosis.top_concerns:
        reinforcement = _join_or_dash(concern.adjustment_reasons)
        lines.append(
            f"- **{concern.label}** finished at **{concern.severity_score:.1f}/100** "
            f"({concern.severity_band}). Base score was **{concern.base_severity_score:.1f}**, "
            f"and external evidence added **{concern.external_adjustment_score:.1f}**. "
            f"Main read: {concern.summary} Reinforcement: {reinforcement}."
        )
    return "\n".join(lines)


def build_top_holding_drivers_df(diagnosis):
    return pd.DataFrame([
        {
            'ticker': driver.ticker,
            'sector': driver.sector,
            'current_weight': driver.current_weight,
            'excess_return_vs_benchmark': driver.excess_return_vs_benchmark,
            'variance_contribution_pct': driver.variance_contribution_pct,
            'primary_reason_label': driver.primary_reason_label,
            'primary_reason_summary': driver.primary_reason_summary,
            'secondary_reasons': _join_or_dash(driver.secondary_reason_codes),
            'driver_confidence_band': driver.driver_confidence_band,
            'driver_confidence_score': driver.driver_confidence_score,
            'key_evidence': _join_or_dash(driver.evidence_summary[:4]),
        }
        for driver in diagnosis.top_holding_drivers
    ])


def build_reason_code_df(diagnosis):
    rows = []
    for driver in diagnosis.top_holding_drivers:
        for reason_code in driver.reason_codes:
            rows.append(
                {
                    'ticker': driver.ticker,
                    'reason_code': reason_code.code,
                    'label': reason_code.label,
                    'category': reason_code.category,
                    'severity_score': reason_code.severity_score,
                    'summary': reason_code.summary,
                    'evidence': _join_or_dash(reason_code.evidence),
                }
            )
    return pd.DataFrame(rows)


def build_holding_driver_markdown(diagnosis):
    lines = ["### What the current top holding drivers mean", ""]
    for driver in diagnosis.top_holding_drivers:
        secondary = _join_or_dash(driver.secondary_reason_codes)
        evidence = _join_or_dash(driver.evidence_summary[:3])
        lines.append(
            f"- **{driver.ticker}** is here mainly because of **{driver.primary_reason_label}**. "
            f"{driver.primary_reason_summary} Secondary support: {secondary}. "
            f"Confidence: **{driver.driver_confidence_band}** ({driver.driver_confidence_score:.2f}). "
            f"Evidence seen: {evidence}."
        )
    return "\n".join(lines)


def build_top_sector_drivers_df(diagnosis):
    return pd.DataFrame([
        {
            'sector': driver.sector,
            'weight_pct': driver.weight_pct,
            'excess_return_vs_benchmark': driver.excess_return_vs_benchmark,
            'primary_reason_label': driver.primary_reason_label,
            'primary_reason_summary': driver.primary_reason_summary,
            'driver_confidence_band': driver.driver_confidence_band,
            'driver_confidence_score': driver.driver_confidence_score,
            'reason_codes': _join_or_dash([code.code for code in driver.reason_codes]),
        }
        for driver in diagnosis.top_sector_drivers
    ])


def build_sector_driver_markdown(diagnosis):
    lines = ["### What the current sector drivers mean", ""]
    for driver in diagnosis.top_sector_drivers:
        lines.append(
            f"- **{driver.sector}** is flagged mainly for **{driver.primary_reason_label}**. "
            f"{driver.primary_reason_summary} Confidence: **{driver.driver_confidence_band}** "
            f"({driver.driver_confidence_score:.2f})."
        )
    return "\n".join(lines)


def build_supporting_evidence_frames(diagnosis):
    supporting_metric_keys = {metric_key for concern in diagnosis.top_concerns for metric_key in concern.evidence_metric_keys}
    supporting_metrics_df = pd.DataFrame([
        metric.model_dump() for metric in diagnosis.supporting_metrics if metric.metric_key in supporting_metric_keys
    ])
    sector_evidence_df = build_top_sector_drivers_df(diagnosis)
    narrative_evidence_df = pd.DataFrame([
        item.model_dump() for item in diagnosis.narrative_evidence
    ])
    macro_context_df = pd.DataFrame([
        diagnosis.macro_context.model_dump() if diagnosis.macro_context else {}
    ])
    return supporting_metrics_df, sector_evidence_df, narrative_evidence_df, macro_context_df


def build_confidence_frames(diagnosis):
    confidence_df = pd.DataFrame([
        {
            'overall_confidence_band': diagnosis.confidence_band,
            'interpretation': confidence_interpretation(diagnosis),
            'alignment': diagnosis.alignment,
            'run_id': diagnosis.run_id,
        }
    ])
    coverage_df = pd.DataFrame([
        {'source': key, 'available': value}
        for key, value in diagnosis.data_coverage.model_dump().items()
    ]).sort_values(['available', 'source'], ascending=[False, True])
    gaps_df = pd.DataFrame([{'evidence_gap': gap} for gap in diagnosis.evidence_gaps])
    return confidence_df, coverage_df, gaps_df



def build_holding_contributions_df(diagnosis):
    return pd.DataFrame([
        {
            'ticker': item.ticker,
            'sector': item.sector,
            'overall_contribution_score': item.overall_contribution_score,
            'overall_contribution_band': item.overall_contribution_band,
            'primary_concern_label': item.primary_concern_label,
            'primary_concern_summary': item.primary_concern_summary,
            'contribution_confidence_band': item.contribution_confidence_band,
            'contribution_confidence_score': item.contribution_confidence_score,
            'contribution_summary': item.contribution_summary,
            'evidence_summary': _join_or_dash(item.evidence_summary),
        }
        for item in diagnosis.holding_risk_contributions
    ])



def build_holding_contribution_detail_df(diagnosis):
    rows = []
    for item in diagnosis.holding_risk_contributions:
        for contribution in item.concern_contributions:
            rows.append(
                {
                    'ticker': item.ticker,
                    'concern_label': contribution.concern_label,
                    'contribution_score': contribution.contribution_score,
                    'contribution_band': contribution.contribution_band,
                    'contribution_summary': contribution.contribution_summary,
                    'supporting_reason_codes': _join_or_dash(contribution.supporting_reason_codes),
                    'supporting_evidence': _join_or_dash(contribution.supporting_evidence),
                    'confidence_band': contribution.confidence_band,
                    'confidence_score': contribution.confidence_score,
                }
            )
    return pd.DataFrame(rows)



def build_holding_contribution_markdown(diagnosis):
    lines = ["### How each holding feeds the diagnosis", ""]
    for item in diagnosis.holding_risk_contributions:
        lines.append(
            f"- **{item.ticker}** is primarily contributing to **{item.primary_concern_label}** "
            f"with an overall contribution score of **{item.overall_contribution_score:.1f}/100**. "
            f"{item.contribution_summary} Confidence: **{item.contribution_confidence_band}** "
            f"({item.contribution_confidence_score:.2f})."
        )
    return "\n".join(lines)



def build_holding_contribution_examples_markdown(diagnosis):
    lines = ["### Examples from the current fake dataset", ""]
    for item in diagnosis.holding_risk_contributions[:3]:
        top_secondary = ", ".join(cc.concern_label for cc in item.concern_contributions[1:3]) or "no major secondary concern"
        lines.append(
            f"- **{item.ticker}**: primary concern is **{item.primary_concern_label}**. "
            f"Why: {item.primary_concern_summary} Secondary spillover: {top_secondary}."
        )
    return "\n".join(lines)


## 1. Overall Diagnosis Summary

This section should answer the user's first trust question:

- What risk did I say I was comfortable with?
- What risk does my portfolio actually look like?
- Are those aligned?
- How confident is the system in that conclusion?


In [3]:

overall_summary_df = build_overall_summary_df(diagnosis)
display(overall_summary_df)
display(Markdown(build_fake_dataset_overview_markdown(diagnosis)))
display(Markdown(f"**Diagnostic summary from the object:** {diagnosis.diagnostic_summary}"))


,observed_risk_score,observed_risk_band,stated_risk_score,stated_risk_band,alignment_status,overall_confidence,confidence_interpretation,analysis_start,analysis_end,benchmark_symbol
0,53.4,Moderate,60.0,Moderate,Aligned,High,High confidence: multiple independent signals ...,2020-08-10,2026-04-01,^GSPC


### Current fake-dataset diagnosis

For the current saved fake-dataset run, the portfolio looks **aligned** with the user's stated risk tolerance: observed risk is **53.4/100 (Moderate)** versus stated risk **60.0/100 (Moderate)**.

The strongest current concerns are **Market-relative risk, Concentration risk, Behavioral risk**. The main holding-level drivers right now are **NVDA (volatility contributor), AAPL (narrative or event risk), MSFT (narrative or event risk)**, while the sector view is led by **Technology, ETF / Fund**.

This means the portfolio is not flashing a pure "overall risk mismatch" warning right now. Instead, the diagnosis is saying: the portfolio still has **specific fragility pockets** that deserve attention, especially in the names and sectors listed below.

**Diagnostic summary from the object:** Observed portfolio risk is 53.4/100 (Moderate) versus a stated risk of 60.0/100 (Moderate). The portfolio currently looks aligned. The main diagnosis is driven by market-relative risk, concentration risk, over the analysis window 2020-08-10 to 2026-04-01. External evidence reinforced this read through macro backdrop still has restrictive rates; sticky inflation can pressure richly priced risk assets.

## 2. Top Risk Categories (Ranked)

The diagnosis should rank what kind of risk is dominant, not just dump scores.

This view combines:
- native diagnosis concerns from the object
- derived sector crowding view
- derived company-specific risk view
- derived macro sensitivity view


In [4]:

top_risk_categories_df = build_top_risk_categories_df(diagnosis)
display(top_risk_categories_df)
display(Markdown(build_top_concern_markdown(diagnosis)))


,label,base_severity_score,external_adjustment_score,severity_score,severity_band,summary,external_reinforcement,related_tickers,related_sectors
0,Market-relative risk,62.6,12.0,74.6,High concern,The portfolio has recently behaved more aggres...,"macro backdrop still has restrictive rates, st...","NVDA, AAPL, MSFT","Technology, ETF / Fund"
1,Concentration risk,66.0,0.0,66.0,High concern,The portfolio looks concentrated in a small nu...,-,"NVDA, AAPL, MSFT","Technology, ETF / Fund"
2,Behavioral risk,9.8,2.5,12.3,Low concern,"Behavioral risk looks relatively contained, bu...",several top drivers are lagging benchmark desp...,-,-


### Why these categories are ranked the way they are

- **Market-relative risk** finished at **74.6/100** (High concern). Base score was **62.6**, and external evidence added **12.0**. Main read: The portfolio has recently behaved more aggressively than the S&P 500. Relative volatility is about 1.43x, relative drawdown depth is about 1.63x, and market sensitivity is about 1.36x versus the benchmark. Reinforcement: macro backdrop still has restrictive rates, sticky inflation can pressure richly priced risk assets, multiple main drivers still have above-market beta, external narrative evidence reinforces market sensitivity concerns.
- **Concentration risk** finished at **66.0/100** (High concern). Base score was **66.0**, and external evidence added **0.0**. Main read: The portfolio looks concentrated in a small number of names. Largest position weight is 18.6% and top five holdings account for about 50.1% of capital. Effective holdings are only about 12.68, so diversification is thinner than the ticker count suggests. Reinforcement: -.
- **Behavioral risk** finished at **12.3/100** (Low concern). Base score was **9.8**, and external evidence added **2.5**. Main read: Behavioral risk looks relatively contained, but it still contributes context to the diagnosis. Turnover is about 2.3% and weighted holding period is about 466 days against a target of 548 days. Reinforcement: several top drivers are lagging benchmark despite low trading churn, behavior should still be interpreted in light of evolving company news.


## 3. Top Risk Drivers and Holding Contributions

This section now answers two related questions:

- which holdings and sectors are showing up as drivers
- how each important holding contributes to specific diagnosis concerns

That contribution view is the bridge from diagnosis into later action logic.


In [5]:

top_holding_drivers_df = build_top_holding_drivers_df(diagnosis)
reason_code_df = build_reason_code_df(diagnosis)
top_sector_drivers_df = build_top_sector_drivers_df(diagnosis)
holding_contributions_df = build_holding_contributions_df(diagnosis)
holding_contribution_detail_df = build_holding_contribution_detail_df(diagnosis)

display(Markdown(build_holding_driver_markdown(diagnosis)))
display(top_holding_drivers_df)
display(Markdown("### Holding-level reason codes"))
display(reason_code_df)
display(Markdown(build_holding_contribution_markdown(diagnosis)))
display(holding_contributions_df)
display(Markdown(build_holding_contribution_examples_markdown(diagnosis)))
display(Markdown("### Concern-by-concern contribution detail"))
display(holding_contribution_detail_df)
display(Markdown(build_sector_driver_markdown(diagnosis)))
display(top_sector_drivers_df)


### What the current top holding drivers mean

- **NVDA** is here mainly because of **Volatility contributor**. NVDA has been one of the biggest contributors to recent portfolio swings. Secondary support: concentration_pressure, high_beta. Confidence: **High** (1.00). Evidence seen: Variance contribution: 32.8%, Current weight: 18.6%, Beta: 2.33.
- **AAPL** is here mainly because of **Narrative or event risk**. Recent filings or news around AAPL contain risk-relevant language worth monitoring. Secondary support: restrictive_rates_sensitivity, volatility_contributor. Confidence: **High** (1.00). Evidence seen: 10-Q filing, Apple ( AAPL ) Postpones Its Smart Home Display Release, Macro flag: rates still restrictive.
- **MSFT** is here mainly because of **Narrative or event risk**. Recent filings or news around MSFT contain risk-relevant language worth monitoring. Secondary support: restrictive_rates_sensitivity, benchmark_lag. Confidence: **High** (1.00). Evidence seen: 10-Q filing, Macro flag: rates still restrictive, Excess return vs benchmark: -30.5%.
- **META** is here mainly because of **Narrative or event risk**. Recent filings or news around META contain risk-relevant language worth monitoring. Secondary support: restrictive_rates_sensitivity, volatility_contributor. Confidence: **High** (1.00). Evidence seen: 10-K filing, Macro flag: rates still restrictive, Variance contribution: 9.0%.
- **GOOGL** is here mainly because of **Narrative or event risk**. Recent filings or news around GOOGL contain risk-relevant language worth monitoring. Secondary support: restrictive_rates_sensitivity, meaningful_position. Confidence: **High** (1.00). Evidence seen: 10-K filing, Investors Are Bullish on Alphabet Stock - Piling Into GOOGL Call Options With Huge Unusual Options Volume, Macro flag: rates still restrictive.

,ticker,sector,current_weight,excess_return_vs_benchmark,variance_contribution_pct,primary_reason_label,primary_reason_summary,secondary_reasons,driver_confidence_band,driver_confidence_score,key_evidence
0,NVDA,Technology,0.1858,1.1762,0.3280,Volatility contributor,NVDA has been one of the biggest contributors ...,"concentration_pressure, high_beta",High,1.0,"Variance contribution: 32.8%, Current weight: ..."
1,AAPL,Technology,0.0784,-0.0661,0.1001,Narrative or event risk,Recent filings or news around AAPL contain ris...,"restrictive_rates_sensitivity, volatility_cont...",High,1.0,"10-Q filing, Apple ( AAPL ) Postpones Its Smar..."
2,MSFT,Technology,0.0594,-0.3054,0.0602,Narrative or event risk,Recent filings or news around MSFT contain ris...,"restrictive_rates_sensitivity, benchmark_lag",High,1.0,"10-Q filing, Macro flag: rates still restricti..."
3,META,Communication Services,0.0585,-0.1158,0.0903,Narrative or event risk,Recent filings or news around META contain ris...,"restrictive_rates_sensitivity, volatility_cont...",High,1.0,"10-K filing, Macro flag: rates still restricti..."
4,GOOGL,Communication Services,0.0609,0.7661,0.0436,Narrative or event risk,Recent filings or news around GOOGL contain ri...,"restrictive_rates_sensitivity, meaningful_posi...",High,1.0,"10-K filing, Investors Are Bullish on Alphabet..."


### Holding-level reason codes

,ticker,reason_code,label,category,severity_score,summary,evidence
0,NVDA,volatility_contributor,Volatility contributor,market_behavior,98.4,NVDA has been one of the biggest contributors ...,Variance contribution: 32.8%
1,NVDA,concentration_pressure,Concentration pressure,positioning,65.0,NVDA is large enough to materially influence t...,Current weight: 18.6%
2,NVDA,high_beta,High beta,fundamentals,60.1,"NVDA tends to move more than the market, which...",Beta: 2.33
3,NVDA,narrative_risk,Narrative or event risk,external_evidence,45.0,Recent filings or news around NVDA contain ris...,10-K filing
4,NVDA,restrictive_rates_sensitivity,Restrictive-rate sensitivity,macro,42.0,NVDA sits in a part of the market that can be ...,Macro flag: rates still restrictive
5,AAPL,narrative_risk,Narrative or event risk,external_evidence,55.0,Recent filings or news around AAPL contain ris...,"10-Q filing, Apple ( AAPL ) Postpones Its Smar..."
6,AAPL,restrictive_rates_sensitivity,Restrictive-rate sensitivity,macro,42.0,AAPL sits in a part of the market that can be ...,Macro flag: rates still restrictive
7,AAPL,volatility_contributor,Volatility contributor,market_behavior,30.0,AAPL has been one of the biggest contributors ...,Variance contribution: 10.0%
8,AAPL,meaningful_position,Meaningful position size,positioning,17.2,AAPL still carries enough capital to matter fo...,Current weight: 7.8%
9,AAPL,benchmark_lag,Lagging benchmark,performance,6.6,AAPL has underperformed the trade-matched S&P ...,Excess return vs benchmark: -6.6%


### How each holding feeds the diagnosis

- **NVDA** is primarily contributing to **Market-relative risk** with an overall contribution score of **100.0/100**. NVDA is primarily a market-relative risk driver, with secondary spillover into concentration risk, macro sensitivity. Confidence: **High** (0.88).
- **META** is primarily contributing to **Market-relative risk** with an overall contribution score of **49.0/100**. META is primarily a market-relative risk driver, with secondary spillover into macro sensitivity, company-specific risk. Confidence: **High** (0.94).
- **AAPL** is primarily contributing to **Market-relative risk** with an overall contribution score of **43.8/100**. AAPL is primarily a market-relative risk driver, with secondary spillover into company-specific risk, macro sensitivity. Confidence: **High** (0.92).
- **MSFT** is primarily contributing to **Company-specific risk** with an overall contribution score of **39.3/100**. MSFT is primarily a company-specific risk driver, with secondary spillover into macro sensitivity, market-relative risk. Confidence: **High** (0.84).
- **GOOGL** is primarily contributing to **Macro sensitivity** with an overall contribution score of **30.7/100**. GOOGL is primarily a macro sensitivity driver, with secondary spillover into company-specific risk, market-relative risk. Confidence: **High** (0.80).

...

### Examples from the current fake dataset

- **NVDA**: primary concern is **Market-relative risk**. Why: NVDA mainly contributes to market-relative risk through volatility contributor. Secondary spillover: Concentration risk, Macro sensitivity.
- **META**: primary concern is **Market-relative risk**. Why: META mainly contributes to market-relative risk through narrative or event risk. Secondary spillover: Macro sensitivity, Company-specific risk.
- **AAPL**: primary concern is **Market-relative risk**. Why: AAPL mainly contributes to market-relative risk through narrative or event risk. Secondary spillover: Company-specific risk, Macro sensitivity.

### Concern-by-concern contribution detail

,ticker,concern_label,contribution_score,contribution_band,contribution_summary,supporting_reason_codes,supporting_evidence,confidence_band,confidence_score
0,NVDA,Market-relative risk,100.0,Very high contribution,NVDA mainly contributes to market-relative ris...,"volatility_contributor, high_beta, narrative_risk","Variance contribution: 32.8%, Beta: 2.33, 10-K...",High,0.96
1,NVDA,Concentration risk,42.9,Meaningful contribution,NVDA mainly contributes to concentration risk ...,concentration_pressure,Current weight: 18.6%,High,0.80
2,NVDA,Macro sensitivity,29.7,Moderate contribution,NVDA mainly contributes to macro sensitivity t...,"high_beta, restrictive_rates_sensitivity","Beta: 2.33, Macro flag: rates still restrictive",High,0.88
3,NVDA,Company-specific risk,18.6,Low contribution,NVDA mainly contributes to company-specific ri...,narrative_risk,10-K filing,High,0.80
4,META,Market-relative risk,39.5,Moderate contribution,META mainly contributes to market-relative ris...,"narrative_risk, volatility_contributor, high_b...","10-K filing, Variance contribution: 9.0%, Beta...",High,1.00
5,META,Macro sensitivity,24.6,Moderate contribution,META mainly contributes to macro sensitivity t...,"restrictive_rates_sensitivity, high_beta","Macro flag: rates still restrictive, Beta: 1.31",High,0.88
6,META,Company-specific risk,23.0,Moderate contribution,META mainly contributes to company-specific ri...,"narrative_risk, benchmark_lag","10-K filing, Excess return vs benchmark: -11.6%",High,0.88
7,META,Concentration risk,5.5,Low contribution,META mainly contributes to concentration risk ...,meaningful_position,Current weight: 5.9%,High,0.80
8,AAPL,Market-relative risk,34.1,Moderate contribution,AAPL mainly contributes to market-relative ris...,"narrative_risk, volatility_contributor, benchm...","10-Q filing, Apple ( AAPL ) Postpones Its Smar...",High,0.96
9,AAPL,Company-specific risk,25.2,Moderate contribution,AAPL mainly contributes to company-specific ri...,"narrative_risk, benchmark_lag","10-Q filing, Apple ( AAPL ) Postpones Its Smar...",High,0.88


### What the current sector drivers mean

- **Technology** is flagged mainly for **Sector crowding**. Technology represents a large enough share of capital to shape portfolio behavior. Confidence: **Medium** (0.54).
- **ETF / Fund** is flagged mainly for **Sector crowding**. ETF / Fund represents a large enough share of capital to shape portfolio behavior. Confidence: **Medium** (0.54).
- **Financial Services** is flagged mainly for **Sector benchmark lag**. Financial Services has lagged the trade-matched benchmark inside this portfolio. Confidence: **Medium** (0.54).

,sector,weight_pct,excess_return_vs_benchmark,primary_reason_label,primary_reason_summary,driver_confidence_band,driver_confidence_score,reason_codes
0,Technology,0.4140,0.5277,Sector crowding,Technology represents a large enough share of ...,Medium,0.54,sector_crowding
1,ETF / Fund,0.2631,0.0341,Sector crowding,ETF / Fund represents a large enough share of ...,Medium,0.54,sector_crowding
2,Financial Services,0.0518,-0.0369,Sector benchmark lag,Financial Services has lagged the trade-matche...,Medium,0.54,sector_benchmark_lag


## 4. Supporting Evidence

This is not raw-data overload. It is the minimum evidence necessary to support the diagnosis:

- benchmark-relative risk metrics
- sector exposure evidence
- filing/news evidence summaries
- macro regime context


In [6]:

supporting_metrics_df, sector_evidence_df, narrative_evidence_df, macro_context_df = build_supporting_evidence_frames(diagnosis)
holding_fundamentals_df = pd.DataFrame([item.model_dump() for item in diagnosis.holding_fundamentals])

display(Markdown("### Supporting risk metrics used by the top concerns"))
display(supporting_metrics_df[['metric_key', 'group', 'label', 'raw_value', 'score', 'score_readout', 'meaning']])
display(Markdown("### Holding fundamental snapshots behind the diagnosis"))
display(holding_fundamentals_df)
display(Markdown("### Narrative evidence currently attached to the main drivers"))
display(narrative_evidence_df)
display(Markdown("### Macro regime context used by the diagnosis"))
display(macro_context_df)
display(Markdown("### Sector evidence snapshot"))
display(sector_evidence_df)


### Supporting risk metrics used by the top concerns

,metric_key,group,label,raw_value,score,score_readout,meaning
0,concentration::single_position_weight,Concentration,Largest position size,0.1858,84.5,Very high concern,Checks whether one holding is large enough to ...
1,concentration::top_5_weight,Concentration,Top 5 holdings dominance,0.5011,77.1,High concern,Checks whether just a few names are carrying m...
2,concentration::effective_holdings,Concentration,True diversification,12.6832,36.6,Mild concern,Looks past ticker count and asks how many hold...
3,behavior::turnover,Behavior,Trading churn,0.0230,4.6,Low concern,Measures how much of the portfolio you rotate ...
4,behavior::short_holding_period,Behavior,How long your dollars stay invested,466.0000,15.0,Low concern,Measures whether larger investments are being ...
5,market::relative_volatility_to_benchmark,Market,Volatility vs S&P 500,1.4273,42.7,Moderate concern,Checks whether your portfolio swings around mo...
6,market::relative_drawdown_to_benchmark,Market,Downside depth vs S&P 500,1.6316,63.2,High concern,Checks whether your portfolio's bad stretches ...
7,market::relative_downside_capture_to_benchmark,Market,Bad-day behavior vs S&P 500,1.3079,100.0,Very high concern,Checks how your portfolio tends to behave on m...
8,market::relative_market_sensitivity_to_benchmark,Market,Market sensitivity vs S&P 500,1.3551,47.3,Moderate concern,Checks how strongly your portfolio tends to mo...
9,market::equity_exposure,Market,How fully invested you are,0.5994,59.9,Moderate concern,Checks how much of your account is currently i...


### Holding fundamental snapshots behind the diagnosis

,ticker,company_name,sector,industry,market_cap,beta,revenue,net_income,cash_and_equivalents,total_assets,total_liabilities,stockholders_equity,operating_cash_flow,latest_filed_date,signals
0,NVDA,NVIDIA Corporation,Technology,Semiconductors,4.584652e+12,2.335,2.159380e+11,1.200670e+11,1.060500e+10,2.068030e+11,3.740000e+08,1.572930e+11,1.027180e+11,2026-02-25,"[above-market beta, latest net income positive..."
1,VOO,Vanguard S&P 500 ETF,Financial Services,Asset Management,1.419877e+12,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,[]
2,AAPL,Apple Inc.,Technology,Consumer Electronics,3.828515e+12,1.109,2.156390e+11,4.209700e+10,4.531700e+10,3.792970e+11,1.198770e+11,8.819000e+10,5.392500e+10,2026-01-30,"[latest net income positive, operating cash fl..."
3,MSFT,Microsoft Corporation,Technology,Software - Infrastructure,2.753943e+12,1.107,2.681900e+10,3.845800e+10,2.429600e+10,6.653020e+11,-1.290100e+10,3.908750e+11,3.575800e+10,2026-01-28,"[latest net income positive, operating cash fl..."
4,META,"Meta Platforms, Inc.",Communication Services,Internet Content & Information,1.587873e+12,1.309,1.372700e+10,6.045800e+10,3.587300e+10,3.660210e+11,2.210000e+09,2.172430e+11,1.158000e+11,2026-01-29,"[above-market beta, latest net income positive..."
5,NOW,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,[]
6,MMYT,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,[]
7,GOOGL,Alphabet Inc.,Communication Services,Internet Content & Information,3.837652e+12,1.128,4.028360e+11,1.321700e+11,3.070800e+10,5.952810e+11,5.122800e+10,4.152650e+11,1.647130e+11,2026-02-05,"[latest net income positive, operating cash fl..."
8,QQQ,"Invesco QQQ Trust, Series 1",Financial Services,Asset Management,3.757478e+11,1.110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,[]
9,TSM,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,[]


### Narrative evidence currently attached to the main drivers

,ticker,source_type,document_date,title,url,snippet
0,NVDA,sec_filing,2026-02-25,10-K filing,https://www.sec.gov/Archives/edgar/data/104581...,Item 1A. Risk Factors &#160; 12
1,AAPL,sec_filing,2026-01-30,10-Q filing,https://www.sec.gov/Archives/edgar/data/320193...,Item 1A. Risk Factors 20
2,AAPL,news_article,None,Apple ( AAPL ) Postpones Its Smart Home Displa...,https://www.insidermonkey.com/blog/apple-aapl-...,Apple Inc. (NASDAQ: AAPL ) earns a place on ou...
3,MSFT,sec_filing,2026-01-28,10-Q filing,https://www.sec.gov/Archives/edgar/data/789019...,Item 1A. Risk Factors 47 &#160; &#160; &#160; ...
4,META,sec_filing,2026-01-29,10-K filing,https://www.sec.gov/Archives/edgar/data/132680...,Item 1A. Risk Factors 12
5,GOOGL,sec_filing,2026-02-05,10-K filing,https://www.sec.gov/Archives/edgar/data/165204...,Risk Factors 9 Item&#160;1B. Unresolved Staff ...
6,GOOGL,news_article,None,Investors Are Bullish on Alphabet Stock - Pili...,https://finance.yahoo.com/news/investors-bulli...,Investors are trading Alphabet Inc ( GOOGL ) c...


### Macro regime context used by the diagnosis

,as_of_date,fed_funds_rate,inflation_yoy,unemployment_rate,ten_year_yield,two_year_yield,yield_curve_spread,regime_flags,summary
0,2026-04-09,3.64,0.03286,4.3,4.29,3.78,0.51,"[rates still restrictive, inflation still stic...",Macro backdrop looks moderately restrictive bu...


### Sector evidence snapshot

,sector,weight_pct,excess_return_vs_benchmark,primary_reason_label,primary_reason_summary,driver_confidence_band,driver_confidence_score,reason_codes
0,Technology,0.4140,0.5277,Sector crowding,Technology represents a large enough share of ...,Medium,0.54,sector_crowding
1,ETF / Fund,0.2631,0.0341,Sector crowding,ETF / Fund represents a large enough share of ...,Medium,0.54,sector_crowding
2,Financial Services,0.0518,-0.0369,Sector benchmark lag,Financial Services has lagged the trade-matche...,Medium,0.54,sector_benchmark_lag


## 5. Confidence and Completeness Flags

A financial diagnosis should never overstate certainty.

This section makes explicit:
- the overall confidence level
- how broad the source coverage is
- where the current diagnosis still has blind spots


In [7]:

confidence_df, coverage_df, gaps_df = build_confidence_frames(diagnosis)
display(confidence_df)
display(Markdown("### Source coverage flags"))
display(coverage_df)
display(Markdown("### Evidence gaps the current diagnosis still admits"))
display(gaps_df)


,overall_confidence_band,interpretation,alignment,run_id
0,High,High confidence: multiple independent signals ...,Aligned,20260419_162540


### Source coverage flags

,source,available
7,company_facts_available,True
8,company_profiles_available,True
5,filing_text_available,True
9,macro_series_available,True
6,news_text_available,True
1,open_positions_available,True
4,performance_attribution_available,True
0,risk_metrics_available,True
2,sector_allocation_available,True
3,volatility_drivers_available,True


### Evidence gaps the current diagnosis still admits

,evidence_gap
0,User goals and constraints beyond the stated r...
1,Fundamental snapshots are based on broad canon...


## Save Current Enriched Diagnosis

This writes the current object view to disk so we can compare revisions over time.


In [8]:
OUTPUT_PATH.write_text(json.dumps(diagnosis.model_dump(), indent=2) + '\n', encoding='utf-8')
print('Saved enriched diagnosis object to:')
print(OUTPUT_PATH)


Saved enriched diagnosis object to:
/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/data/processed/diagnosis/portfolio_risk_diagnosis_enriched.json



## Review Prompts

When reviewing this notebook, useful questions are:

- Do the ranked categories feel believable?
- Are the named risk drivers actually the right ones?
- Does the new `HoldingRiskContribution` layer feel like a real bridge from diagnosis into action?
- Are the primary vs secondary concern assignments believable for each holding?
- Is the supporting evidence too thin, too noisy, or just right?
- Does the confidence section feel appropriately humble?
- What should be surfaced directly in the UI, and what should remain secondary?
